In [1]:
# ============================================================
# RQ3 FEATURE 1 — Significance Test (Paired Bootstrap)
# Restaurant-only vs Restaurant+Hotel+Food checkpoints
# Evaluated on 7 languages, restaurant domain test sets
# ============================================================

# ------------------------------------------------------------
# 1. Install packages
# ------------------------------------------------------------
!pip install transformers datasets evaluate seqeval -q

# ------------------------------------------------------------
# 2. Imports
# ------------------------------------------------------------
import os, re, json
import pandas as pd
import numpy as np
from datasets import Dataset
import evaluate
from transformers import (
    AutoModelForTokenClassification, AutoTokenizer,
    Trainer, DataCollatorForTokenClassification
)

seqeval = evaluate.load("seqeval")

# ------------------------------------------------------------
# 3. Mount Drive + load data
# ------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_parquet("/content/drive/MyDrive/MABSA_Shared_Project/data/master_dataset.parquet")
print("df loaded:", df.shape)

# ------------------------------------------------------------
# 4. Helper functions
# ------------------------------------------------------------
def make_bio_tags(sentence, aspect, category, sentiment, task="tasd"):
    tokens = sentence.split()
    tags = ["O"] * len(tokens)
    if pd.isna(aspect) or str(aspect).strip() in ("", "NULL"):
        return tokens, tags
    aspect_tokens = aspect.split()
    n = len(aspect_tokens)

    def strip_punct(w):
        return re.sub(r'^[.,!?;:"\'()\[\]،]+|[.,!?;:"\'()\[\]،]+$', '', w)

    clean_tokens = [strip_punct(t) for t in tokens]
    clean_aspect_tokens = [strip_punct(t) for t in aspect_tokens]

    for i in range(len(clean_tokens) - n + 1):
        if clean_tokens[i:i+n] == clean_aspect_tokens:
            tags[i] = f"B-{category}-{sentiment}"
            for j in range(1, n):
                tags[i+j] = f"I-{category}-{sentiment}"
            break
    return tokens, tags

def build_tagged_df(source_df, task="tasd"):
    records = []
    for _, row in source_df.iterrows():
        tokens, tags = make_bio_tags(row.sentence, row.aspect, row.category, row.sentiment, task=task)
        records.append({"tokens": tokens, "tags": tags, "split": row.split})
    return pd.DataFrame(records)

def make_tokenize_fn_safe(label2id_local, tok):
    def tokenize_and_align_labels(examples):
        tokenized_inputs = tok(examples["tokens"], truncation=True, is_split_into_words=True)
        all_labels = []
        for i, tags in enumerate(examples["tags"]):
            word_ids = tokenized_inputs.word_ids(batch_index=i)
            label_ids, prev = [], None
            for word_idx in word_ids:
                if word_idx is None:
                    label_ids.append(-100)
                elif word_idx != prev:
                    label_ids.append(label2id_local.get(tags[word_idx], -100))
                else:
                    label_ids.append(-100)
                prev = word_idx
            all_labels.append(label_ids)
        tokenized_inputs["labels"] = all_labels
        return tokenized_inputs
    return tokenize_and_align_labels

def make_compute_metrics(label_list_local):
    def compute_metrics(p):
        preds = np.argmax(p.predictions, axis=2)
        true_labels = [[label_list_local[l] for l in label if l != -100] for label in p.label_ids]
        true_preds = [[label_list_local[pp] for (pp, l) in zip(pred, label) if l != -100]
                      for pred, label in zip(preds, p.label_ids)]
        r = seqeval.compute(predictions=true_preds, references=true_labels)
        return {"f1": r["overall_f1"], "precision": r["overall_precision"], "recall": r["overall_recall"]}
    return compute_metrics

def get_label_sequences(ckpt_dir, lang_ds):
    """Returns (true_labels, pred_labels) as lists of label-string sequences,
    aligned per example, for use in paired bootstrap testing."""
    model = AutoModelForTokenClassification.from_pretrained(ckpt_dir)
    tok = AutoTokenizer.from_pretrained(ckpt_dir)
    with open(os.path.join(ckpt_dir, "label_map.json")) as f:
        label2id = json.load(f)
    id2label = {v: k for k, v in label2id.items()}

    tok_fn = make_tokenize_fn_safe(label2id, tok)
    collator = DataCollatorForTokenClassification(tok)
    trainer = Trainer(model=model, processing_class=tok, data_collator=collator)

    tokenized = lang_ds.map(tok_fn, batched=True)
    preds = trainer.predict(tokenized)
    pred_ids = np.argmax(preds.predictions, axis=2)

    true_labels = [[id2label[l] for l in label if l != -100] for label in preds.label_ids]
    pred_labels = [[id2label[pp] for pp, l in zip(pred, label) if l != -100]
                   for pred, label in zip(pred_ids, preds.label_ids)]
    return true_labels, pred_labels

def paired_bootstrap_test(true_labels_a, pred_labels_a, true_labels_b, pred_labels_b,
                            n_bootstrap=1000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(true_labels_a)
    assert n == len(true_labels_b), "Both systems must be evaluated on the same test set"

    r_a = seqeval.compute(predictions=pred_labels_a, references=true_labels_a)
    r_b = seqeval.compute(predictions=pred_labels_b, references=true_labels_b)
    f1_a, f1_b = r_a["overall_f1"], r_b["overall_f1"]
    observed_diff = f1_a - f1_b

    diffs = []
    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, n)
        sample_true_a = [true_labels_a[i] for i in idx]
        sample_pred_a = [pred_labels_a[i] for i in idx]
        sample_true_b = [true_labels_b[i] for i in idx]
        sample_pred_b = [pred_labels_b[i] for i in idx]

        r_a_boot = seqeval.compute(predictions=sample_pred_a, references=sample_true_a)
        r_b_boot = seqeval.compute(predictions=sample_pred_b, references=sample_true_b)
        diffs.append(r_a_boot["overall_f1"] - r_b_boot["overall_f1"])

    diffs = np.array(diffs)
    if observed_diff >= 0:
        p_value = 2 * np.mean(diffs <= 0)
    else:
        p_value = 2 * np.mean(diffs >= 0)
    p_value = min(p_value, 1.0)
    ci_lower, ci_upper = np.percentile(diffs, [2.5, 97.5])

    return {
        "f1_a": f1_a, "f1_b": f1_b, "observed_diff": observed_diff,
        "p_value": p_value, "ci_diff_95": (float(ci_lower), float(ci_upper))
    }

# ------------------------------------------------------------
# 5. Build language test sets (restaurant domain, punctuation-fixed)
# ------------------------------------------------------------
language_iso_map = {
    "swahili": "sw", "arabic": "ar", "hindi": "hi", "vietnamese": "vi",
    "spanish": "es", "german": "de", "thai": "th"
}
languages = list(language_iso_map.keys())

test_sets = {}
for lang_name, iso in language_iso_map.items():
    lang_test_df = df[(df.language == iso) & (df.split == "test") & (df.domain == "restaurant")].reset_index(drop=True)
    lang_tagged = build_tagged_df(lang_test_df, task="tasd").reset_index(drop=True)

    non_null = lang_test_df[lang_test_df.aspect.notna() & (lang_test_df.aspect.astype(str).str.strip() != "NULL")]
    sample_idx = non_null.index[:50]
    sample_check = lang_tagged.loc[sample_idx]
    unmatched = sum(1 for t in sample_check["tags"] if all(x == "O" for x in t))
    print(f"{lang_name}: {unmatched}/{len(sample_idx)} unmatched")

    test_sets[lang_name] = Dataset.from_pandas(lang_tagged[["tokens", "tags"]])

# ------------------------------------------------------------
# 6. Get label sequences from BOTH checkpoints, per language
#    UPDATE THESE PATHS if your checkpoint folder names differ
# ------------------------------------------------------------
CKPT_A = "/content/drive/MyDrive/MABSA_Shared_Project/models/xlmr_restaurant_only_final"
CKPT_B = "/content/drive/MyDrive/MABSA_Shared_Project/models/xlmr_restaurant_hotel_food_final"

significance_results = {}
for lang in languages:
    print(f"\n=== {lang} ===")
    true_a, pred_a = get_label_sequences(CKPT_A, test_sets[lang])
    true_b, pred_b = get_label_sequences(CKPT_B, test_sets[lang])

    result = paired_bootstrap_test(true_a, pred_a, true_b, pred_b, n_bootstrap=1000)
    significance_results[lang] = result

    sig_flag = "SIGNIFICANT (p<0.05)" if result["p_value"] < 0.05 else "not significant"
    print(f"  Restaurant-only F1: {result['f1_a']:.4f}")
    print(f"  Restaurant+Hotel+Food F1: {result['f1_b']:.4f}")
    print(f"  Observed diff (A-B): {result['observed_diff']:+.4f}")
    print(f"  95% CI of diff: ({result['ci_diff_95'][0]:+.4f}, {result['ci_diff_95'][1]:+.4f})")
    print(f"  p-value: {result['p_value']:.4f} — {sig_flag}")

# ------------------------------------------------------------
# 7. Save results
# ------------------------------------------------------------
out_path = "/content/drive/MyDrive/MABSA_Shared_Project/results/rq3_feature1_significance_test.json"
os.makedirs(os.path.dirname(out_path), exist_ok=True)
with open(out_path, "w") as f:
    json.dump(significance_results, f, indent=2)
print(f"\nSaved to {out_path}")

# ------------------------------------------------------------
# 8. Summary table
# ------------------------------------------------------------
print("\n=== Summary ===")
print(f"{'Language':12s} | {'Rest-only F1':13s} | {'RHF F1':10s} | {'Diff':8s} | {'p-value':8s} | Significant?")
print("-" * 85)
for lang in languages:
    r = significance_results[lang]
    sig = "Yes" if r["p_value"] < 0.05 else "No"
    print(f"{lang:12s} | {r['f1_a']:.4f}        | {r['f1_b']:.4f}     | {r['observed_diff']:+.4f}  | {r['p_value']:.4f}   | {sig}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00


Mounted at /content/drive
df loaded: (438341, 8)
swahili: 0/50 unmatched
arabic: 3/50 unmatched
hindi: 0/50 unmatched
vietnamese: 0/50 unmatched
spanish: 0/50 unmatched
german: 0/50 unmatched
thai: 3/50 unmatched

=== swahili ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Map:   0%|          | 0/788 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Map:   0%|          | 0/793 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Map:   0%|          | 0/793 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


  Restaurant-only F1: 0.1971
  Restaurant+Hotel+Food F1: 0.1695
  Observed diff (A-B): +0.0277
  95% CI of diff: (-0.0072, +0.0571)
  p-value: 0.1100 — not significant

Saved to /content/drive/MyDrive/MABSA_Shared_Project/results/rq3_feature1_significance_test.json

=== Summary ===
Language     | Rest-only F1  | RHF F1     | Diff     | p-value  | Significant?
-------------------------------------------------------------------------------------
swahili      | 0.2126        | 0.1665     | +0.0461  | 0.0040   | Yes
arabic       | 0.2765        | 0.2709     | +0.0056  | 0.7580   | No
hindi        | 0.3019        | 0.2656     | +0.0364  | 0.0300   | Yes
vietnamese   | 0.1541        | 0.1963     | -0.0421  | 0.0080   | Yes
spanish      | 0.3640        | 0.3699     | -0.0059  | 0.7340   | No
german       | 0.3903        | 0.3687     | +0.0216  | 0.2060   | No
thai         | 0.1971        | 0.1695     | +0.0277  | 0.1100   | No
